<a href="https://colab.research.google.com/github/idontknowbr0/Clipfarming/blob/main/AI_Stream_Clip_Finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
import json
import os
import re
from pydub import AudioSegment
from pydub.silence import split_on_silence
import webvtt
from io import StringIO

# --- Configuration ---
# Add keywords that you think are interesting. The script will look for these.
INTERESTING_KEYWORDS = [
    "wow", "omg", "what", "crazy", "insane", "pog", "wait", "really",
    "laughing", "lol", "lmao", "look", "check this out", "holy"
]
# Loudness is measured in dBFS (decibels relative to full scale).
# A lower number (e.g., -16) is stricter and will only find very loud moments.
# A higher number (e.g., -25) is more lenient.
LOUDNESS_THRESHOLD_DBFS = -20
# Minimum length for a clip to be considered, in seconds.
MIN_CLIP_DURATION_S = 5
# How much time to add before and after a detected event (in seconds).
CLIP_PADDING_S = 2
# Directory to save the final clips.
OUTPUT_DIR = "clips"

# --- New Export Configuration (from your clipstream.sh) ---
# Set to True to re-encode clips into an editor-friendly format (H.264/AAC).
# This requires FFmpeg to be installed.
REENCODE_FOR_PREMIERE = True
# The format yt-dlp should try to download for the clips.
# The `/best` is a fallback if the preferred format isn't available.
YTDLP_DOWNLOAD_FORMAT = "bestvideo[height=1080][fps=60]+bestaudio/best"


def time_to_seconds(time_obj):
    """Converts a time string (HH:MM:SS.ms) to seconds."""
    return time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second + time_obj.microsecond / 1_000_000

def seconds_to_ffmpeg_time(seconds):
    """Converts seconds to HH:MM:SS.ms format for ffmpeg."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    ms = int((seconds * 1000) % 1000)
    return f"{hours:02}:{minutes:02}:{secs:02}.{ms:03}"

def get_video_info(url):
    """
    Downloads audio and transcript using yt-dlp, but not the video itself yet.
    Returns paths to the downloaded files.
    """
    print("--- Step 1: Fetching video information, audio, and transcript ---")

    # Ensure temporary directory exists
    temp_dir = "temp"
    os.makedirs(temp_dir, exist_ok=True)

    # Download transcript
    transcript_path_template = os.path.join(temp_dir, "video_subtitles")
    subprocess.run([
        "yt-dlp",
        "--write-auto-sub",
        "--sub-lang", "en",
        "--skip-download",
        "-o", transcript_path_template,
        url
    ], check=True, capture_output=True)

    # Find the downloaded .vtt file
    transcript_file = None
    for file in os.listdir(temp_dir):
        if file.startswith("video_subtitles") and file.endswith(".vtt"):
            transcript_file = os.path.join(temp_dir, file)
            break

    if not transcript_file:
        print("Warning: Could not find a transcript for this video.")

    # Download audio as WAV for analysis
    audio_path_template = os.path.join(temp_dir, "video_audio.%(ext)s")
    subprocess.run([
        "yt-dlp",
        "-f", "bestaudio",
        "-x", "--audio-format", "wav",
        "-o", audio_path_template,
        url
    ], check=True, capture_output=True)

    audio_file = os.path.join(temp_dir, "video_audio.wav")

    if not os.path.exists(audio_file):
        raise FileNotFoundError("Audio file was not downloaded correctly.")

    print(f"Successfully downloaded audio to '{audio_file}'")
    if transcript_file:
        print(f"Successfully downloaded transcript to '{transcript_file}'")

    return audio_file, transcript_file


def analyze_transcript(transcript_file):
    """
    Analyzes a .vtt transcript file for interesting keywords.
    Returns a list of timestamps (start, end) for potential clips.
    """
    if not transcript_file:
        return []

    print("--- Step 2: Analyzing transcript for keywords ---")
    segments = []
    try:
        vtt = webvtt.read(transcript_file)
        for caption in vtt:
            for keyword in INTERESTING_KEYWORDS:
                if re.search(r'\b' + keyword + r'\b', caption.text, re.IGNORECASE):
                    start_s = time_to_seconds(caption.start_obj)
                    end_s = time_to_seconds(caption.end_obj)
                    segments.append({
                        "start": start_s,
                        "end": end_s,
                        "reason": f"Keyword: '{keyword}'"
                    })
                    # Found a keyword in this caption, no need to check for others
                    break
    except Exception as e:
        print(f"Could not parse transcript file: {e}")
        return []

    print(f"Found {len(segments)} potential clips based on keywords.")
    return segments

def analyze_audio(audio_file):
    """
    Analyzes the audio file for moments of high volume.
    Returns a list of timestamps (start, end) for potential clips.
    """
    print("--- Step 3: Analyzing audio for loud moments ---")
    segments = []
    try:
        audio = AudioSegment.from_wav(audio_file)

        # Find loud parts of the audio
        loud_parts = split_on_silence(
            audio,
            min_silence_len=500, # ms
            silence_thresh=LOUDNESS_THRESHOLD_DBFS - 10, # A bit more lenient for silence detection
            keep_silence=250
        )

        # We don't have timestamps yet, so we calculate them
        current_time_ms = 0
        for part in loud_parts:
            # Check if this part itself is loud enough
            if part.dBFS > LOUDNESS_THRESHOLD_DBFS:
                start_s = current_time_ms / 1000.0
                end_s = (current_time_ms + len(part)) / 1000.0
                segments.append({
                    "start": start_s,
                    "end": end_s,
                    "reason": f"Loud Moment ({part.dBFS:.1f} dBFS)"
                })
            current_time_ms += len(part)

    except Exception as e:
        print(f"Could not process audio file: {e}")
        return []

    print(f"Found {len(segments)} potential clips based on volume.")
    return segments

def merge_and_filter_clips(all_segments):
    """
    Merges overlapping or adjacent clip segments and filters out short ones.
    """
    if not all_segments:
        return []

    print("--- Step 4: Merging and filtering potential clips ---")

    # Sort segments by start time
    sorted_segments = sorted(all_segments, key=lambda x: x['start'])

    merged = []
    if not sorted_segments:
        return merged

    current_clip = sorted_segments[0]

    for next_clip in sorted_segments[1:]:
        # If the next clip starts before or very shortly after the current one ends, merge them
        if next_clip['start'] <= current_clip['end'] + CLIP_PADDING_S:
            current_clip['end'] = max(current_clip['end'], next_clip['end'])
            # Combine reasons
            if next_clip['reason'] not in current_clip['reason']:
                current_clip['reason'] += f", {next_clip['reason']}"
        else:
            merged.append(current_clip)
            current_clip = next_clip

    merged.append(current_clip)

    # Add padding and filter by duration
    final_clips = []
    for clip in merged:
        start = max(0, clip['start'] - CLIP_PADDING_S)
        end = clip['end'] + CLIP_PADDING_S
        if end - start >= MIN_CLIP_DURATION_S:
            final_clips.append({
                "start": start,
                "end": end,
                "reason": clip['reason']
            })

    print(f"Identified {len(final_clips)} final clips to download.")
    return final_clips

def download_clips(url, clips):
    """
    Downloads the final list of clips using yt-dlp and optionally re-encodes them.
    """
    print("--- Step 5: Downloading final clips ---")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for i, clip in enumerate(clips):
        start_time = seconds_to_ffmpeg_time(clip['start'])
        end_time = seconds_to_ffmpeg_time(clip['end'])

        reason_slug = re.sub(r'[^a-zA-Z0-9]', '_', clip['reason']).lower()

        # Define temporary and final filenames
        temp_filename_template = os.path.join(OUTPUT_DIR, f"clip_{i+1:02d}_temp.%(ext)s")
        final_filename = os.path.join(OUTPUT_DIR, f"clip_{i+1:02d}_{reason_slug}.mp4")

        print(f"\nDownloading clip {i+1}/{len(clips)}: {clip['reason']}")
        print(f"Time: {start_time} -> {end_time}")

        temp_filepath_actual = None
        try:
            # Step 1: Download the clip to a temporary file
            print("Downloading raw clip...")
            yt_dlp_command = [
                "yt-dlp",
                "-f", YTDLP_DOWNLOAD_FORMAT,
                "--download-sections", f"*{start_time}-{end_time}",
                "-o", temp_filename_template,
                "--force-keyframes-at-cuts",
                "--quiet", "--no-warnings",
                url
            ]
            subprocess.run(yt_dlp_command, check=True)

            # Find the actual downloaded temp file since yt-dlp adds the extension
            temp_file_base = os.path.basename(temp_filename_template).split('.')[0]
            found_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith(temp_file_base)]
            if not found_files:
                raise FileNotFoundError("yt-dlp did not create the expected temp file.")
            temp_filepath_actual = os.path.join(OUTPUT_DIR, found_files[0])

            # Step 2: Re-encode with FFmpeg if enabled
            if REENCODE_FOR_PREMIERE:
                print(f"🎞️ Converting to Premiere-safe MP4 -> {final_filename}")
                ffmpeg_command = [
                    "ffmpeg",
                    "-y",
                    "-i", temp_filepath_actual,
                    "-vf", "format=yuv420p",
                    "-c:v", "libx264",
                    "-preset", "veryfast",
                    "-crf", "18",
                    "-c:a", "aac",
                    "-b:a", "320k",
                    final_filename
                ]
                # Use capture_output to hide verbose ffmpeg logs unless there's an error
                subprocess.run(ffmpeg_command, check=True, capture_output=True)
                print("✅ Re-encode complete.")
            else:
                # If not re-encoding, just rename the temp file to the final name
                os.rename(temp_filepath_actual, final_filename)
                temp_filepath_actual = None # Prevent deletion in finally block
                print("✅ Download complete.")

        except subprocess.CalledProcessError as e:
            print(f"ERROR: A subprocess failed for clip {i+1}.")
            # Provide ffmpeg output if it was the one that failed
            if "ffmpeg" in e.cmd:
                print(f"FFmpeg error:\n{e.stderr.decode() if e.stderr else 'No stderr output.'}")
            else:
                print("yt-dlp failed during download.")
        except Exception as e:
            print(f"An unexpected error occurred during processing clip {i+1}: {e}")
        finally:
            # Step 3: Clean up the temporary raw download
            if temp_filepath_actual and os.path.exists(temp_filepath_actual):
                os.remove(temp_filepath_actual)


def cleanup():
    """Removes temporary files."""
    print("--- Cleaning up temporary files ---")
    temp_dir = "temp"
    if os.path.exists(temp_dir):
        for file in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, file))
        os.rmdir(temp_dir)
    print("Cleanup complete.")

if __name__ == "__main__":
    import sys
    if len(sys.argv) < 2:
        print("Usage: python clip_finder.py <youtube_or_twitch_url>")
        sys.exit(1)

    video_url = sys.argv[1]

    try:
        # 1. Get audio and transcript
        audio_file, transcript_file = get_video_info(video_url)

        # 2. Analyze both
        transcript_segments = analyze_transcript(transcript_file)
        audio_segments = analyze_audio(audio_file)

        # 3. Combine and process results
        all_potential_clips = transcript_segments + audio_segments
        final_clips_to_download = merge_and_filter_clips(all_potential_clips)

        # 4. Download the clips
        if final_clips_to_download:
            download_clips(video_url, final_clips_to_download)
        else:
            print("\nNo clips found that match the criteria.")

    except FileNotFoundError as e:
        print(f"\nERROR: A required file was not found. Please check dependencies. Details: {e}")
    except subprocess.CalledProcessError as e:
        print(f"\nERROR: yt-dlp failed. Make sure it's installed and the URL is correct.")
        print(f"yt-dlp output:\n{e.stderr.decode() if e.stderr else 'No stderr output.'}")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")
    finally:
        # 5. Clean up temp files
        cleanup()